# Langfuse Observability — Callback-Based LLM Tracing
## Traces in Production — Langfuse Callbacks for LangGraph
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/77-langfuse-observability/langfuse_observability_workbook.ipynb)

Instruments a LangGraph pipeline with a callback-based trace handler that mirrors the Langfuse interface. Runs fully in Colab without Langfuse credentials; shows how to swap in the real handler for production.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Observability layers: LLM calls, nodes, full runs; Langfuse vs LangSmith |
| 2 | **SimpleTraceHandler** — BaseCallbackHandler capturing on_llm_start / on_llm_end |
| 3 | **LangGraph + Callbacks** — Pass handler via callbacks= on ChatOpenAI |
| 4 | **Trace Inspection** — Print captured events; show trace_id per task |
| 5 | **Langfuse Swap-In** — One-line change to use real CallbackHandler in production |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
import uuid
from typing import Any, TypedDict
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

SAMPLE_TASKS = [
    "What is machine learning in one sentence?",
    "Explain neural networks in one sentence.",
    "What is gradient descent in one sentence?",
]

class SimpleTraceHandler(BaseCallbackHandler):
    """Mirrors the Langfuse CallbackHandler interface — works in Colab without credentials."""
    def __init__(self, trace_id: str):
        super().__init__()
        self.trace_id = trace_id
        self.events: list[dict] = []

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.events.append({"event": "llm_start", "trace_id": self.trace_id, "prompt": prompts[0][:80]})

    def on_llm_end(self, response, **kwargs):
        text = response.generations[0][0].text if response.generations else ""
        self.events.append({"event": "llm_end",   "trace_id": self.trace_id, "output": text[:80]})

class TraceState(TypedDict):
    task: str; answer: str; trace_id: str

def make_graph(handler: SimpleTraceHandler):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, callbacks=[handler])
    def ask_question(state: TraceState) -> dict:
        return {**state, "answer": llm.invoke([HumanMessage(content=state["task"])]).content.strip()}
    def format_answer(state: TraceState) -> dict:
        return {**state, "answer": f"[{state['trace_id'][:8]}] {state['answer']}"}
    g = StateGraph(TraceState)
    g.add_node("ask_question", ask_question)
    g.add_node("format_answer", format_answer)
    g.add_edge(START, "ask_question"); g.add_edge("ask_question", "format_answer"); g.add_edge("format_answer", END)
    return g.compile()

for task in SAMPLE_TASKS:
    trace_id = str(uuid.uuid4())
    handler = SimpleTraceHandler(trace_id)
    app = make_graph(handler)
    result = app.invoke({"task": task, "answer": "", "trace_id": trace_id})
    print(f"Task:  {task}")
    print(f"A:     {result['answer']}")
    print(f"Trace events: {len(handler.events)}")
    for e in handler.events:
        print(f"  [{e['event']}] {e.get('prompt', e.get('output', ''))[:70]}")
    print()

print("To use real Langfuse in production:")
print("  pip install langfuse")
print("  from langfuse.callback import CallbackHandler")
print("  handler = CallbackHandler(public_key='...', secret_key='...', host='https://cloud.langfuse.com')")
print("  # Rest of code unchanged — just swap SimpleTraceHandler for CallbackHandler")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*